# Myllia - echoes of silenced genes
---

**authors**: [fsb2210](https://www.kaggle.com/fsb2210), [julianc93](https://www.kaggle.com/julianc93)

## Introduction

In this notebok, we compute the embeddings of all the protein sequences that represent the perturbation genes we are given (`pert_symbol`) using a state-of-the-art neural network model **ESM-2** from *Meta-AI*.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

Define global options:

In [ ]:
# random state integer value for reproducibility concerns
random_state = 42

# pre-trained neural network for gene embeddings
model_name = "facebook/esm2_t33_650M_UR50D"
device = "cuda" if torch.cuda.is_available() else "cpu"

# filename with embeddings from training and validation datasets
output_filename = "esm2_gene_embeddings.csv"

In [ ]:
# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device);

## Data loading

In [ ]:
# train & validation datasets
raw_train_df = pd.read_csv("/kaggle/input/datasets/mylliabiotechnology/myllia-echoes-of-silenced-genes-competition-data/training_data_means.csv")
valid_df = pd.read_csv("/kaggle/input/datasets/mylliabiotechnology/myllia-echoes-of-silenced-genes-competition-data/pert_ids_val.csv")

In [ ]:
# K562 dataset
raw_k562_df = pd.read_csv("/kaggle/input/datasets/fsb2210/scrna-seq-processed-data-means/k562_data_means.csv")
# Hepg2 dataset
raw_hepg2_df = pd.read_csv("/kaggle/input/datasets/fsb2210/scrna-seq-processed-data-means/hepg2_data_means.csv")
# Jurkat dataset
raw_jurkat_df = pd.read_csv("/kaggle/input/datasets/fsb2210/scrna-seq-processed-data-means/jurkat_data_means.csv")
# RPE1 dataset
raw_rpe1_df = pd.read_csv("/kaggle/input/datasets/fsb2210/scrna-seq-processed-data-means/rpe1_data_means.csv")

In [ ]:
prot_sequence_df = pd.read_csv("/kaggle/input/datasets/fsb2210/scrna-seq-processed-data-means/uniprotkb_organism_id_9606_2026_02_26.tsv", sep="\t").dropna(axis=0)
prot_sequence_df["split_genes"] = prot_sequence_df["Gene Names"].str.split()
prot_sequence_df.shape

### Preprocessing steps

Grab the name of the perturbations in each of the datasets

In [ ]:
train_pert_genes = set(raw_train_df.pert_symbol.tolist()[:-1])
k562_pert_genes = set(raw_k562_df.pert_symbol.tolist()[:-1])
hepg2_pert_genes = set(raw_hepg2_df.pert_symbol.tolist()[:-1])
jurkat_pert_genes = set(raw_jurkat_df.pert_symbol.tolist()[:-1])
rpe1_pert_genes = set(raw_rpe1_df.pert_symbol.tolist()[:-1])

In [ ]:
train_genes = train_pert_genes
train_genes.update(k562_pert_genes)
train_genes.update(hepg2_pert_genes)
train_genes.update(jurkat_pert_genes)
train_genes.update(rpe1_pert_genes)

Grab the name of the output genes:

In [ ]:
output_genes = set(raw_train_df.columns[1:])

Get the list of genes that **must** be present in the network

In [ ]:
valid_genes = set(valid_df.pert.tolist())

In [ ]:
final_genes = train_genes
final_genes.update(output_genes)
final_genes.update(valid_genes)

## Feature and target engineering

Now we are ready to treat every of the protein sequences (`pert_symbol`) as tokens and use trained protein languange model to retrieve their embeddings such that genes with similar molecular roles end up having similar embeddings.

In this case, we try with *ESM-2* from Meta-AI (`esm2_t33_650M_UR50D`) from the transformers library to get embeddings of size 1280.

> NOTE: we use float16 (`autocast` from PyTorch) for a fast evaluation of embeddings. 

But first, we start by getting the sequence representation of each of the genes in the `final_genes` set:

In [ ]:
gene_to_entry = {}
for idx, row in prot_sequence_df.iterrows():
    for gene in row["split_genes"]:
        if gene not in gene_to_entry:
            gene_to_entry[gene] = row["Sequence"]

In [ ]:
# get sequences for each of the genes
seqs = []
failed = []
for gene in final_genes:
    if gene in gene_to_entry:
        seqs.append((gene, gene_to_entry[gene]))
    else:
        failed.append(gene)

In [ ]:
print(f"success: {len(seqs)}/{len(final_genes)}, failed: {len(failed)}")

Split sequences based on how long they are:

In [ ]:
seqs_short  = [(g, s) for g, s in seqs_filtered if len(s) <= 500]
seqs_medium = [(g, s) for g, s in seqs_filtered if 500 < len(s) <= 2000]
seqs_long = [(g, s) for g, s in seqs if len(s) > 2000]
len(seqs_short), len(seqs_medium), len(seqs_long)

Now we can work on their embeddings. We use `Dataset` and `DataLoader` for a batched lookup of protein sequences to be sent to the `ESM-2` model

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, seq_tuples):
        # sort by sequence length descending to minimize padding
        self.seq_tuples = sorted(seq_tuples, key=lambda x: len(x[1]), reverse=True)

    def __len__(self):
        return len(self.seq_tuples)

    def __getitem__(self, idx):
        gene, seq = self.seq_tuples[idx]
        return gene, seq

In [ ]:
from torch.cuda.amp import autocast

def esm_collate_fn(batch, max_length):
    genes, seqs = zip(*batch)
    inputs = tokenizer(seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    return inputs, list(genes)

def get_esm_embeddings(seq_tuples, tokenizer, model, device, batch_size=8, max_length=1024) -> dict:
    """Batch ESM embeddings for list of (gene, sequence) tuples."""
    dataset = ProteinDataset(seq_tuples)
    collate = lambda b: esm_collate_fn(b, max_length=max_length)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=4, pin_memory=True)
    
    model.eval()
    embeddings = {}
    with torch.no_grad():
        for batch_tokens, genes in tqdm(data_loader, desc="ESM2 embeddings"):
            batch_tokens = {k: v.to(device) for k, v in batch_tokens.items()}
            with autocast():
                outputs = model(**batch_tokens)
            # outputs = model(**batch_tokens)
            last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
            residue_embs = last_hidden[:, 1:-1, :]
            pooled = residue_embs.mean(dim=1).cpu().numpy()
            for i, gene in enumerate(genes):
                embeddings[gene] = pooled[i]
    return embeddings

- Shorter protein sequences

In [ ]:
emb_short  = get_esm_embeddings(seqs_short, tokenizer, model, device, batch_size=128, max_length=512)

- Medium size protein sequences

In [ ]:
emb_medium = get_esm_embeddings(seqs_medium, tokenizer, model, device, batch_size=32,  max_length=2048)

* Long size protein sequences

In [ ]:
emb_long = get_esm_embeddings(seqs_long, tokenizer, model, device, batch_size=4, max_length=2048)

Now we combine them into a single embedding dict:

In [ ]:
embeddings = {}
embeddings.update(emb_short)
embeddings.update(emb_medium)
embeddings.update(emb_long)

Be very careful that genes in the validation set all have embeddings. If not, we will not be able to submit them to the competition!

In [ ]:
for gene in valid_genes:
    if gene not in embeddings.keys(): print(f"{gene} not found")

Save embeddings as a `DataFrame`

In [ ]:
emb_df = pd.DataFrame.from_dict(embeddings, orient="index", columns=range(len(embeddings["ACLY"])))
emb_df.insert(0, "pert_symbol", embeddings.keys())
emb_df.reset_index(drop=True)
emb_df.to_csv(output_filename, index=False)